In [1]:
import dspy
import os
api_key = os.environ.get("GEMINI_API_KEY")

lm = dspy.LM("gemini/gemini-2.5-flash", api_key=api_key)
dspy.configure(lm=lm)

In [2]:
math = dspy.ChainOfThought("question -> answer: float")
math(question="Two dice are tossed. What is the probability that the sum equals two?")

Prediction(
    reasoning="When two dice are tossed, the total number of possible outcomes is 6 * 6 = 36.\nEach die can show a number from 1 to 6.\nWe are looking for the outcomes where the sum of the two dice equals two.\nThe only way to get a sum of two is if both dice show a 1.\nSo, the only favorable outcome is (1, 1).\nNumber of favorable outcomes = 1.\nThe probability is the ratio of the number of favorable outcomes to the total number of possible outcomes.\nProbability = 1 / 36.\n\nTo convert this to a float: 1 / 36 = 0.027777...\nRounding to a few decimal places, for example, 0.0278 or keeping the exact fraction representation for high precision. The prompt asks for a float, so calculating it directly.\n\n1 / 36 = 0.027777777777777776 (in Python)\nRounding to a common precision like 4 or 5 decimal places: 0.02778 or 0.027778.\nLet's use the full precision Python float result for maximum accuracy if not specified.\nThe question doesn't specify rounding, so I will provide the dir

In [3]:
import wikipedia

# 1. Scrape the specific Wikipedia page
def get_wiki_context(url: str) -> list[str]:
    # Extract page title from URL: "https://en.wikipedia.org/wiki/Royal_Albert_Hall" -> "Royal Albert_Hall"
    page_title = url.split('/')[-1]
    
    # Fetch content using the wikipedia library
    page = wikipedia.page(page_title, auto_suggest=False)
    content = page.content
    
    # Chunking: Split by double newlines (paragraphs) to create a list of strings
    # DSPy contexts usually work best as a list of strings
    chunks = [p.strip() for p in content.split('\n\n') if len(p.strip()) > 100]
    return chunks

In [4]:
rag = dspy.ChainOfThought("context, question -> response")

royal_albert_hall_url = "https://en.wikipedia.org/wiki/Royal_Albert_Hall"
context_chunks = get_wiki_context(royal_albert_hall_url)

question = "In what year did the Royal Albert Hall open?"
prediction = rag(context=context_chunks, question=question)

print(f"Question: {question}")
print(f"Answer: {prediction.response}")

Question: In what year did the Royal Albert Hall open?
Answer: The Royal Albert Hall opened in 1871.


In [5]:
question = "How long was the Royal Albert Hall closed for during COVID?"
prediction = rag(context=context_chunks, question=question)

print(f"Question: {question}")
print(f"Answer: {prediction.response}")

Question: How long was the Royal Albert Hall closed for during COVID?
Answer: The Royal Albert Hall was initially closed on 17 March 2020. After reopening for three socially distanced performances in December 2020 and a subsequent second closure, it finally reopened to full capacity on 19 July 2021. This indicates an overall closure period from March 17, 2020, to July 19, 2021, which is approximately 1 year and 4 months, with a brief intermission in December 2020.


# GameInterpreter

In [6]:
from urllib.request import urlopen

url = "https://raw.githubusercontent.com/zczlsde/GameInterpreter/a0ffa6e9c2ca486ab7af270fea23186f8b39b639/Prompts/Code_Generation_Initialization.txt"

with urlopen(url) as response:
    # urlopen returns bytes, so you must decode it to a string
    file_contents = response.read().decode('utf-8')

print(file_contents)

Given a game description in natural language, you will be asked to generate python code for the Gambit API (pygambit) to construct a corresponding extensive-form game in Gambit. 
Here are two examples of how to use pygambit library:

Example 1:
Game description:
There are two players, a Buyer and a Seller. The Buyer moves first and has two actions, Trust or Not trust. If the Buyer chooses Not trust, then the game ends, and both players receive payoffs of 0. If the Buyer chooses Trust, then the Seller has a choice with two actions, Honor or Abuse. If the Seller chooses Honor, both players receive payoffs of 1; if the Seller chooses Abuse, the Buyer receives a payoff of -1 and the Seller receives a payoff of 2.
Code:
```python
import pygambit as gbt
g = gbt.Game.new_tree(players=["Buyer", "Seller"],
                    title="One-shot trust game, after Kreps (1990)")

g.append_move(g.root, "Buyer", ["Trust", "Not trust"])
g.append_move(g.root.children[0], "Seller", ["Honor", "Abuse"])
g.

In [7]:
game_desc = "Both players simultaneously reveal one of three symbols: rock, paper, or scissors. Rock defeats scissors by blunting it, scissors defeat paper by cutting it, and paper defeats rock by covering it."
prediction = rag(context=file_contents, question=game_desc)
prediction.response

'import pygambit as gbt\ng = gbt.Game.new_tree(players=["Player 1", "Player 2"],\n                    title="Rock-Paper-Scissors")\n\n# Player 1\'s move\ng.append_move(g.root, "Player 1", ["Rock", "Paper", "Scissors"])\n\n# Player 2\'s move for each of Player 1\'s choices\n# Create a list to hold the nodes where Player 2 makes a decision\nplayer2_decision_nodes = []\nfor p1_choice_node in g.root.children:\n    g.append_move(p1_choice_node, "Player 2", ["Rock", "Paper", "Scissors"])\n    player2_decision_nodes.append(p1_choice_node)\n\n# Set information sets for Player 2 to simulate simultaneous move\n# All nodes where Player 2 makes a choice are grouped into one information set.\n# The first node\'s infoset is used as the reference.\nfor i in range(1, len(player2_decision_nodes)):\n    g.set_infoset(player2_decision_nodes[i], player2_decision_nodes[0].infoset)\n\n# Define outcomes (Payoff for P1, Payoff for P2)\ndraw = g.add_outcome([0, 0], label="Draw")\np1_wins = g.add_outcome([1, -1

In [8]:
for line in prediction.response.split("\\n"):
    print(line)

import pygambit as gbt
g = gbt.Game.new_tree(players=["Player 1", "Player 2"],
                    title="Rock-Paper-Scissors")

# Player 1's move
g.append_move(g.root, "Player 1", ["Rock", "Paper", "Scissors"])

# Player 2's move for each of Player 1's choices
# Create a list to hold the nodes where Player 2 makes a decision
player2_decision_nodes = []
for p1_choice_node in g.root.children:
    g.append_move(p1_choice_node, "Player 2", ["Rock", "Paper", "Scissors"])
    player2_decision_nodes.append(p1_choice_node)

# Set information sets for Player 2 to simulate simultaneous move
# All nodes where Player 2 makes a choice are grouped into one information set.
# The first node's infoset is used as the reference.
for i in range(1, len(player2_decision_nodes)):
    g.set_infoset(player2_decision_nodes[i], player2_decision_nodes[0].infoset)

# Define outcomes (Payoff for P1, Payoff for P2)
draw = g.add_outcome([0, 0], label="Draw")
p1_wins = g.add_outcome([1, -1], label="Player 1 Wins")
p